# Homework 5: Data collection

AI and in particular Deep Learning have taken the world by storm in recent years. Whenever you think about Deep Learning, you should think about *data*. To train a deep learning model, you always need data. 

text-to-image diffusion models such as Stable-Diffusion are trained on datasets such as [LAION-5B](https://laion.ai/blog/laion-5b/), which contains more than 5 **billion** images, taking up about 500GB of memory! Think about all the engineering challenges that come with loading such amounts of data into a GPU for deep learning... 

In many cases, collecting the data is not enough: you need to 'label' the data, to show the model what it is supposed to do. This is a branch of machine learning called Supervised Learning, as you might have already learned in other courses such as *Machine Learning*.

In this homework we will prepare for training such a deep learning model ourselves. The task at hand? Learning to detect keypoints on the microplate that you will use for the final assignment of this course. Instead of having to manually click the corresponding points in two images, as you did in the previous practical, we will train a deep learning model to do this. 

Below you can see the microplate we want to detect, and the four keypoints we will use. (can you already think of a convenient object frame?)

![](images/microplate-keypoints.png)


There are three main steps to create datasets:
1. collect images
2. annotate the images
3. preprocess the dataset to the appropriate format (many deep learning models use a different format for the datasets)

You will tackle all these steps in this homework.



## Preparations

<div class="alert alert-block alert-info"> <b>Task 1:</b> Run the lines below to  verify your installation.</div>



In [ ]:
!current_env=$(conda info --envs | grep "*" | awk '{print $1}'); if [ "$current_env" = "irm" ]; then echo "The current active Conda environment is 'irm'. You can run the next cell."; else echo "The current active Conda environment is not 'irm'. Make sure to run this notebook with your irm conda env (conda activate irm)"; fi

## Image Collection.

In this step, you will collect images of the microplate. We want our model to generalize to different backgrounds. You might have already collected some images during the practicals (no worry if you didn't) but we want our data to be as diverse as possible to increase generalization performance.
To this end, make sure to use a variety of backgrounds (put it on your kitchen table, your desk, the floor, ... Be creative and make sure to maximize diversity). Aim for 10 different backgrounds/scenes. You can also add different objects in the scene if you want to.

We also want our model to generalize to different poses of the microplate and/or camera, as well. To this end, aim to take 5 pictures from different angles for each scene.

<div class="alert alert-block alert-info"> <b>Task 2:</b> Take between 50 and 100 images of the microplate using the above guidelines.</div>






<div class="alert alert-block alert-info"> <b>Task 3:</b> Put these images as well as a diverse selection of the ones you took at the end of the previous practical session in a folder `data/images` relative to the notebook. In total you should have between 150 and 200 images. </div>

your file structure should look like this:
```
/
homework.ipynb
data/
    images/ 
        <your images here>
```

## Labeling the images using CVAT

To label the images, we will use a labeling tool. There are many tools available, but we will use [cvat](https://www.cvat.ai/), This tool is open-source and has many features.



### CVAT setup

<div class="alert alert-block alert-info"> <b>Task 4:</b> create an account on cvat and log in </div>

You can create an account (one for each group suffices) [here](https://app.cvat.ai/auth/login).

Also Watch [this](https://youtu.be/uI2OEoR08ME?si=0a5GKDxvqq5mqnZV&t=53) video to get acquainted with Projects and Tasks in cvat. You only need to watch untill 1.38.




<div class="alert alert-block alert-info"> <b>Task 5:</b>  create a new project in cvat </div>

Follow the steps below.

- pick IRM as project name
-  select `raw` for the configuration
- paste the next snippet into the configuration

```json
[
  {
    "name": "microplate.bbox",
    "id": 1594481,
    "color": "#7abc06",
    "type": "rectangle",
    "attributes": []
  },
  {
    "name": "microplate.top_left",
    "id": 1594482,
    "color": "#000000",
    "type": "any",
    "attributes": []
  },
  {
    "name": "microplate.bottom_left",
    "id": 1594483,
    "color": "#ff0000",
    "type": "points",
    "attributes": []
  },
  {
    "name": "microplate.top_right",
    "id": 1594484,
    "color": "#0000ff",
    "type": "any",
    "attributes": []
  },
  {
    "name": "microplate.bottom_right",
    "id": 1594485,
    "color": "#24b353",
    "type": "any",
    "attributes": []
  }
]
```
The steps are summarized in this image:


![](images/cvat_instructions/project-creation.png)



If you now open this newly created project, it should look as follows:

![image.png](images/cvat_instructions/project-overview.png)

<div class="alert alert-block alert-info"> <b>Task 6:</b>  Create a new task in this project. </div>

Next, we will create a **Task** in this project.
Fill in the details as below:
![](images/cvat_instructions/task-creation.png)
And drag your images into the field at the bottom, click `Submit & Open` to upload (this will take some time).

<div class="alert alert-block alert-info"> <b>Task 7:</b>  Enable autosave on CVAT </div>


Click on your profile in the top right corner, then press `Settings`.
Now enable auto-save every five minutes to make sure you don't lose annotations:

![](images/cvat_instructions/autosave.png)

### Labeling 


Now you can start labeling the images (click on the Job). 

Watch [this video](https://ugentbe.sharepoint.com/:v:/t/EA06AIRO-ClusterFrancis-03CourseIRM/EQRUPtkYOY1Jiyum3kUtmh4BnFL8AdkkN8MhUbEb6TrvPA?e=UFzPnX), which will show you how to label a single image.

Make sure to zoom in to accurately mark the keypoint! And first label the keypoints, only then the bounding box.

Tip: it is faster to select one keypoint and then label all images, instead of switching every time. 
Tip: learn the shortcuts! use N to start a new annotation and F to go to the next image.


<div class="alert alert-block alert-info"> <b>Task 8:</b>  Label the images </div>


We can now download the annotations, follow the instructions below:

![](images/cvat_instructions/export.png)

<div class="alert alert-block alert-info"> <b>Task 9:</b>  Download the annotations and unzip them. Copy the `annotations.xml` file into your `data` folder. </div>


## Data Preprocessing

Almost there. We now only need to make sure the dataset is in the right format.

### Converting the dataset to COCO format

There exist many popular dataset formats. We will use [COCO](https://cocodataset.org/#format-data) as this is needed for the keypoint detection framework we will use next time.

To get acquainted with the format, answer the following questions using the information on the [coco website](https://cocodataset.org/#format-data):

<div class="alert alert-block alert-info"> <b>Task 10:</b> Fill in the questions about the COCO format</div>


1. Each keypoint has a visibility flag (`v`). How many values are possible for this flag?

<b>Answer here</b>

2. What does each value mean? 


<b>Answer here</b>

<div class="alert alert-block alert-info"> <b>Task 11:</b>  Run the cell below to convert the images from the CVAT format to the COCO format </div>
Make sure there are no errors. If there are, check your cvat annotations: does each image have exactly one keypoint for each category (three in total)?

In [ ]:
import pathlib
import tqdm 
filepath = pathlib.Path.cwd()

!airo-dataset-tools convert-cvat-to-coco-keypoints {filepath / 'data'/'annotations.xml'} {filepath/'data'/'coco_categories.json'}

# change the base directory for the images to the parent file of the annotations.json file
import json

with open(filepath/'data'/'annotations.json', 'r') as f:
    obj = json.load(f)

    for img_dict in tqdm.tqdm(obj['images']):
        img_dict['file_name'] = "images/"+ img_dict['file_name']

with open(filepath/'data'/'annotations.json', 'w') as f:
    json.dump(obj, f)


### Resizing the Dataset
The images you took with your smartphone need to be resized to 1024x512 images. Of course we also need to resize the annotations (which are stored in pixel coordinates). 
We have a tool for this in the airo-mono repo.


<div class="alert alert-block alert-info"> <b>Task 12:</b> Run the cell below to resize the dataset </div>


In [ ]:
# resize the images to 1024x512
!airo-dataset-tools resize-coco-dataset --width 1024 --height 512 {filepath/'data'/'annotations.json'}

## Visualizing the dataset

It is important to manually inspect the data after preprocessing. Many subtle issues can arise, and your machine learning model won't warn you abou these issues by throwing an error, *instead it will silenty fail* and have a diminished performance. This is part of the reason that software engineering for machine learning can be more tricky than regular software engineering. If you want to read more about this, we  recommend [this blogpost](https://karpathy.github.io/2019/04/25/recipe/) from Andrej Karpathy.

Again, we have a command line interface that allows you to quickly visualize a dataset. 

<div class="alert alert-block alert-info"> <b>Task 13:</b> Run the cell below to visualize the resized dataset </div>

Click the url to open a tab in your browers. Make sure the keypoints look correct, as in the examples below:
![](images/cvat_instructions/fo-examples.png)


In [ ]:
!airo-dataset-tools fiftyone-coco-viewer {filepath/'annotations_resized_1024x512'/'annotations.json'} -l keypoints

## Preparation for the Keypoint Detection model.

In the practical session, we will use this dataset to train a model for keypoint detection. As this requires a GPU, we will make use of [google colab](https://colab.research.google.com/). If you don't want to have a google account, you can contact us (at least two days before the practical) and we will find another solution.

We will also make use of a tool for experiment tracking: [weights and biases](https://wandb.ai/) (aka `wandb`).

<div class="alert alert-block alert-info"> <b>Task 14:</b>  Use the link above to create an account on wandb </div>

<div class="alert alert-block alert-info"> <b>Task 15:</b>  Uncomment and run the cells below to install wandb and to log in </div>



In [ ]:
# !pip install wandb

In [ ]:
#get your api key at https://wandb.ai/authorize
YOUR_API_KEY = "fill_in_here"

import wandb 
wandb.login(key=YOUR_API_KEY)


In [ ]:
# cell below should output 'currently logged in as: <your username>'
!wandb login

You are now ready for the Practical session! Make sure to bring your dataset with you.